# Laborator

## Regresie Liniara

### Cerință

Se dau temperaturile in Baia Mare la diferite ore timp de 12 ore...
Task-ul principal este să prezicem temperatura peste o oră față de momentul prezent, pe baza temperaturilor cunoscute.

Date:
- Temperatură (&deg;C): `[15, 18, 20, 22, 25, 27, 30, 32, 35, 37, 40, 38]`
- Ora: `[12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]`
- Temperaturile observate sunt între 15 și 40 (&deg;C).
- Ore față de prezent: 1 = acum o oră, 0 = acum, -1 = peste o oră.

Sarcini:
1. Vizualizarea datelor pentru a înțelege natura relației.
2. Antrenarea unui model de regresie liniară simplă.
3. Calcularea metricilor de evaluare: MAE, MSE, RMSE și R<sup>2</sup>.
4. Predicție pentru peste o oră (`hours_ago = -1`) folosind modelul antrenat.
5. Discuție despre adecvarea modelului liniar (underfitting) și validitatea predicției.

### Rezolvare

Importăm bibliotecile necesare.

In [ ]:
import matplotlib.pyplot as plt  # pentru grafice
import numpy as np  # pentru calcul algebric
from sklearn.linear_model import LinearRegression  # algoritmul de regresie liniară
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # metrici de evaluare

Definim datele cu care vom lucra (din enunțul problemei).

In [ ]:
hours_ago = np.array([12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]).reshape(-1, 1)  # numărul de ore înainte de prezent
temperatures = np.array([15, 18, 20, 22, 25, 27, 30, 32, 35, 37, 40, 38])  # temperaturile observate, în °C

Trasăm grafic punctele din textul problemei pentru vizualizare.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(hours_ago, temperatures, color='blue', label='Data points')
plt.xlabel('Hours Ago')
plt.ylabel('Temperature (°C)')
plt.title('Temperature vs Hours Ago')
plt.grid(True)
plt.legend()
plt.gca().invert_xaxis()
plt.show()

Creăm și antrenăm modelul.

In [ ]:
model = LinearRegression()
model.fit(hours_ago, temperatures)

Afișăm coeficienții rezultați după antrenare.

In [ ]:
coef = model.coef_[0]
intercept = model.intercept_
print(f"Model Coefficients:")
print(f"  Coefficient: {coef:.2f}")
print(f"  Intercept: {intercept:.2f}")

Regresia liniară este descrisă de o funcție de gradul 1 (o dreaptă). După antrenare aceasta este $f(x) = 43.23 - 2.30 * x$.

Generăm predicțiile, calculăm metricile pe datele de antrenare și afișăm graficul. Predicția pentru `hours_ago = -1` este o extrapolare.

In [ ]:
predictions = model.predict(hours_ago)

mae = mean_absolute_error(temperatures, predictions)
mse = mean_squared_error(temperatures, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(temperatures, predictions)

print("=== Linear Regression Metrics ===")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

plt.figure(figsize=(8, 6))
plt.scatter(hours_ago, temperatures, color='blue', label='Data points')
plt.plot(hours_ago, predictions, color='red', label='Linear fit')
plt.xlabel('Hours Ago')
plt.ylabel('Temperature (°C)')
plt.title('Linear Regression Fit')
plt.grid(True)
plt.legend()
plt.gca().invert_xaxis()
plt.show()

next_hour = np.array([[-1]])  # peste o oră față de prezent
prediction_next = model.predict(next_hour)[0]
print(f"\nPrediction one hour from now (hours_ago = -1): {prediction_next:.2f} °C")


## Regresie logistică

Regresia logistică estimează probabilitatea unei clase; aici folosim pragul 0,5 pentru clasificare.

### Cerință

Dorim să prezicem dacă un student va promova un examen în funcție de orele de studiu.

Date:
- Ore studiate: `[1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]`
- Promovat (1=Da, 0=Nu): `[0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]`

Sarcini:
1. Vizualizarea datelor
2. Antrenarea modelelor de regresie logistică
3. Trasarea curbei sigmoid
4. Calcularea matricii de confuzie, acurateții, preciziei, recall-ului și scorului F1
5. Predicție pentru un student care a studiat 4 ore
6. Găsirea orelor de studiu necesare pentru 90% șanse de a promova

### Rezolvare

Importăm bibliotecile necesare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression  # algoritm de clasificare binară
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score  # metrici de evaluare
import seaborn as sns  # aspect grafice

Definim datele cu care vom lucra (din enunțul problemei).

In [ ]:
hours = np.array([1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]).reshape(-1, 1)  # orele de studiu
passed = np.array([0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])  # promovat: 1 = da, 0 = nu

Afișăm grafic datele problemei.

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.scatter(hours, passed, color='blue', s=100, alpha=0.6, edgecolors='black', linewidths=1.5)
plt.xlabel('Hours Studied', fontsize=12)
plt.ylabel('Passed (0=No, 1=Yes)', fontsize=12)
plt.title('Student Exam Results', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.yticks([0, 1], ['Failed', 'Passed'])

Creăm și antrenăm modelul.

In [ ]:
model = LogisticRegression()
model.fit(hours, passed)

Afișăm formula rezultată după antrenare și coeficienții acesteia.

In [ ]:
b0 = model.intercept_[0]
b1 = model.coef_[0][0]
print(f"Model Coefficients:")
print(f"  Intercept (b0): {b0:.4f}")
print(f"  Coefficient (b1): {b1:.4f}")
print(f"\nModel equation: P(pass) = 1 / (1 + e^(-({b0:.4f} + {b1:.4f} * hours)))")

Generăm predicțiile și afișăm graficul funcției.

In [ ]:
plt.figure(figsize=(14, 5))

hours_range = np.linspace(0, 11, 300).reshape(-1, 1)
probabilities = model.predict_proba(hours_range)[:, 1]

y_pred = model.predict(hours)

plt.subplot(1, 2, 2)
plt.scatter(hours, passed, color='blue', s=100, alpha=0.6, label='Actual',
            edgecolors='black', linewidths=1.5, zorder=3)
plt.plot(hours_range, probabilities, color='red', linewidth=3, label='Sigmoid curve', zorder=2)
plt.axhline(y=0.5, color='green', linestyle='--', linewidth=2, label='Probability threshold (0.5)', zorder=1)
plt.xlabel('Hours Studied', fontsize=12)
plt.ylabel('Probability of Passing', fontsize=12)
plt.title('Logistic Regression Model', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Generăm matricea de confuzie. `labels=[0, 1]` păstrează forma 2×2 inclusiv pentru un subset cu o singură clasă. Un asemenea subset nu evaluează recunoașterea celeilalte clase.

In [ ]:
cm = confusion_matrix(passed, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print(f"\n{'=' * 50}")
print("CONFUSION MATRIX")
print(f"{'=' * 50}")
print(f"\n                Predicted")
print(f"              Fail (0)  Pass (1)")
print(f"Actual Fail    {tn:4d}      {fp:4d}")
print(f"Actual Pass    {fn:4d}      {tp:4d}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Predicted Fail', 'Predicted Pass'],
            yticklabels=['Actual Fail', 'Actual Pass'],
            annot_kws={"size": 16})
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual Class', fontsize=12)
plt.xlabel('Predicted Class', fontsize=12)
plt.tight_layout()
plt.show()

Calculăm și afișăm metricile pe datele de antrenare.

In [ ]:
accuracy = accuracy_score(passed, y_pred)
precision = precision_score(passed, y_pred)
recall = recall_score(passed, y_pred)
f1 = f1_score(passed, y_pred)

print(f"{'=' * 50}")
print("PERFORMANCE METRICS")
print(f"{'=' * 50}")
print(f"True Positives (TP):   {tp}")
print(f"True Negatives (TN):   {tn}")
print(f"False Positives (FP):  {fp}")
print(f"False Negatives (FN):  {fn}")
print(f"\nAccuracy:  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1-Score:  {f1:.4f}")

Prezentăm interpretarea metricilor.

In [ ]:
print(f"{'=' * 50}")
print("INTERPRETATION")
print(f"{'=' * 50}")
print(f"Accuracy:  {accuracy * 100:.2f}% of all predictions are correct")
print(f"Precision: {precision * 100:.2f}% of predicted passes are actual passes")
print(f"Recall:    {recall * 100:.2f}% of actual passes were correctly identified")
print(f"F1-Score:  {f1:.4f} (harmonic mean of precision and recall)")

Calculăm predicția pentru 4 ore de studiu.

In [ ]:
hours_4 = np.array([[4]])
prob_4 = model.predict_proba(hours_4)[0, 1]
pred_4 = model.predict(hours_4)[0]

print(f"{'=' * 50}")
print("PREDICTION FOR 4 HOURS OF STUDY")
print(f"{'=' * 50}")
print(f"Probability of passing: {prob_4:.4f} ({prob_4 * 100:.2f}%)")
print(f"Prediction: {'PASS' if pred_4 == 1 else 'FAIL'}")

Calculăm orele de studiu necesare pentru o șansă de 90% de a promova.

In [ ]:
hours_90 = (np.log(0.9 / 0.1) - b0) / b1
prob_90_check = model.predict_proba([[hours_90]])[0, 1]

print(f"{'=' * 50}")
print("HOURS NEEDED FOR 90% PASS PROBABILITY")
print(f"{'=' * 50}")
print(f"Hours needed: {hours_90:.2f}")
print(f"Verification: {prob_90_check:.4f} ({prob_90_check * 100:.2f}%)")

Calculăm și afișăm probabilitățile de promovare pentru diferite ore de studiu.

In [ ]:
print(f"{'=' * 50}")
print("PASS PROBABILITIES FOR DIFFERENT STUDY HOURS")
print(f"{'=' * 50}")
for h in [2, 4, 6, 8]:
    prob = model.predict_proba([[h]])[0, 1]
    print(f"{h} hours: {prob:.4f} ({prob * 100:.2f}%)")

# Temă

**Sarcină:** Rezolvați și prezentați problemele următoare similar cu cele rezolvate la laborator.

Studenții de la specializarea Matematică informatică vor lucra în grupe de câte două persoane.

## Regresie Liniară
Un proprietar de cafenea vrea să prezică vânzările zilnice în funcție de numărul de clienți care intră pe zi.

Date:

Număr clienți: `[50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160]`

Vânzări (lei): `[200, 240, 280, 320, 360, 400, 440, 480, 520, 560, 600, 640]`

Sarcini:

1. Vizualizarea datelor pentru a înțelege natura relației.
2. Antrenarea unui model de regresie liniară simplă.
3. Calcularea metricilor de evaluare: MAE, MSE, RMSE și R².
4. Predicție pentru o zi cu 170 clienți folosind modelul antrenat.
5. Discuție despre adecvarea modelului liniar (underfitting) și validitatea predicției.

## Regresie logistică

Dorim să prezicem șansele de boală cardiacă în funcție de vârstă și nivel de colesterol.

Date:
- Vârstă: `[25, 30, 35, 40, 45, 50, 52, 55, 58, 60, 62, 65, 68, 70, 72, 75, 78, 80]`
- Colesterol (mg/dL): `[180, 190, 195, 200, 210, 220, 235, 240, 250, 255, 265, 270, 280, 285, 295, 300, 310, 320]`
- Boală cardiacă (1=Da, 0=Nu): `[0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]`

Sarcini:
1. Împărțire stratificată în seturi de antrenament (14 puncte) și testare (4 puncte), folosind `train_test_split` cu `test_size=4`, `stratify=y` și `random_state=42`.
2. Antrenare model de regresie logistică
3. Calcularea tuturor metricilor pentru seturile de date de antrenare și testare
4. Vizualizarea funcției de decizie
5. Prezicerea probabilității de boală pentru o persoană de 55 de ani cu nivelul de colesterol 260 mg/dl
6. Discutarea importanței fiecărui feature (caracteristici)